Thiết kế Planner
Goal: 
So sánh 3 đối thủ cạnh tranh của OpenAI theo product, pricing, và positioning
; viết report có bằng chứng cho từng so sánh.

Subtask 1:
Xác định 3 đối thủ cạnh tranh phù hợp (VD: Anthropic, Google DeepMind,
Mistral - hoặc theo tiêu chí tụ thể nếu user có chỉ định thêm).

Subtask 2 [entity=A]:
Research Competitor A - thu nhập product, pricing, positioning.

Subtask 3 [entity=B]:
Research Competitor B - thu nhập product, pricing, positioning.

Subtask 4 [entity=C]:
Research Competitor C - thu nhập product, pricing, positioning.

Subtask 5:
Chuản hóa dữ liệu 3 đối thủ về cùng schema/đơn vị so sánh được.

Subtask 6:
So sánh chéo 3 đối thủ theo từng chiều (product/pricing/positioning)

Subtask 7:
Viết report cuối cùng, trích dẫn nguồn cho mỗi so sánh


Dependencies:
1 -> {2,3,4} -> 5 -> 6 -> 7

Exit Condition:
Cả 3 đối thủ đều có đủ 3 trường dữ liệu (product, pricing, positioning)
với ít nhất 1 nguồn đáng tin cậy mỗi trường, và report cuối có trích dẫn nguồn cho từng nguồn so sánh cụ thể



Strategy A: Plan First
Planner:

Goal: 
So sánh 3 đối thủ OpenAI theo product/pricing/positioning, viết report.

Plan:
1. Identify 3 competitors
2. Research Competitor A (product, pricing, positioning)
3. Research Competitor B (product, pricing, positioning)
4. Research Competitor C (product, pricing, positioning)
5. Normalize data
6. Comparing findings
7. Write report

Executor:

Step 1
Thought: Cần xác định 3 đối thủ cụ thể trước khi research.
Action: web_search(query="top OpenAI competitors 2026")
Observation: Anthropic, Google DeepMind, Mistral AI được nhắc tới nhiều nhất.

Step 2
Thought: Theo plan, research Competitor A (Anthropic) trước.
Action: web_search(query="Anthropic pricing product positioning")
Observation: Tìm thấy trang web Anthropic + trang pricing Claude.

Step 3
Thought: Mở trang pricing để lấy chi tiết.
Action: open_page(url="anthropic.com/pricing")
Observation: [nội dung trang pricing]

Step 4
Thought: Rút trích structure data từ trang.
Action: extract_information(content=..., fields=["product", "pricing", "positioning"])
Observation: {product: "Claude models", pricing: "$3-15/M tokens", positioning="safety-focused"}

Step 5-7: Lặp lại cấu trúc step 2-4 cho competitor B (DeepMind)
Step 8-10: Lặp lại cấu trúc cho competitor C (Mistral)

Step 11
Thought: Đã có đủ dữ liệu 3 đối thủ, chuẩn hóa để so sánh.
Action: (normalize - logic nội bộ, có thể không cần tool call)
Observation: Dữ liệu 3 công ty đã ở cùng định dạng.

Step 12
Thought: So sánh chéo 3 đối thủ theo từng chiều.
Action: (compare - reasoning nội bộ)
Observation: Bảng so sánh hoàn chỉnh.

Step 13
Thought: Viết report cuối.
Action: write_report(data=comparison_table)
Observation: Report hoàn chỉnh, có trích dẫn nguồn


In [ ]:
# Strategy B: Replanning
plan = planner.create_plan(goal)

while not plan.is_complete():
    action = plan.next_action()
    observation = execute(action)
    plan = replan(goal, plan, observation)

return generate_answer()


In [ ]:
MAX_ITERATIONS = 30
MAX_TOTAL_REPLANS = 5
MAX_REPLANS_PER_SUBSTASK = 2
MAX_TOOL_RETRIES = 2

def run_agent(goal):
    plan = planner.create_plan(goal)
    state = {
        "iteration": 0,
        "total_replans": 0,
        "replans_per_subtask": {},
        "trace": [],
    }

    while not plan.is_complete():
        state["iteration"] += 1

        if state["iteration"] > MAX_ITERATIONS:
            log(state, "stop_reason=max_iterations")
            return generate_answer(plan, partial=True)

        task = plan.next_task()

        observation = execute_with_retry(task, max_retries=MAX_TOOL_RETRIES)
        state["trace"].append({"task": task, "observation": observation})

        if observation.is_tool_error and observation.retries_exhausted:
            plan.mark_failed(task, reason="tool_unavailable")
            continue

        replan_scope = decide_replan_scope(plan, task, observation)

        if replan_scope == "none":
            plan.mark_completed(task)

        elif replan_scope == "subtask":
            count = state["replans_per_subtask"].get(task.id, 0)
            if count >= MAX_REPLANS_PER_SUBSTASK:
                plan.accept_best_effor(task, confidence="low")
                log(state, f"subtask {task.id}: replan budget exhausted")
            else:
                state["replans_per_subtask"][task.id] = count + 1
                plan = planner.replan(goal, plan, observation, scope="subtask")
                state["total_replans"] += 1

        elif replan_scope == "global":
            if state["total_replans"] >= MAX_TOTAL_REPLANS:
                log(state, "stop_reason=max_total_replans -> fallback to plan-first")
                plan.freeze()
            else:
                state["total_replans"] += 1
                plan = planner.replan(goal, plan, observation, scope="global")

        if is_oscillating(state["trace"]):
            log(state, "stop_reason=oscillation_detected")
            plan.freeze()

    return generate_answer(plan)


Tình huống buộc Agent phải Replan

Initial plan:
1. Identify competitors -> Anthropic, GooglDeepMind, Mistral
2. Research Anthropic (product, pricing, positioning)
3. Research Google DeepMind (product, pricing, positioning)
4. Research Mistral (product, pricing, positioning)
5. Normalize -> 6. Compare -> 7. Write Report

Tool Execution(Step "Research Google DeepMind - pricing"):
Action: web_search(query="Google DeepMind pricing")
Obsevation: 
"Google DeepMind không phải sản phẩm bán trực tiếp - mô hình được
tích hợp qua Google Cloud Vertex AI, Gemeni API. Không có trang 
pricing riêng cho DeepMind, mà là pricing của Gemeni API,
nằm rải rác theo nhiều tier (Flash/Pro), khác cấu trúc hoàn toàn so với subscription phẳng của Anthropic

Unexpected Observation:
Cấu trúc pricing của Competitor B không tương thích trực tiếp với schema 
ban đầu (không phải 1 con số/tháng, mà nhiều tier theo API usage)

Initial Plan no longer optimal:
Subtask 5 "Normalize data" ban đầu giả định 3 đối thủ có cấu trúc 
pricing tương tự nhau (dễ quy về 1 định dạng) - giả định này SAI với
Competitor B, nên nếu tiếp tục plan gốc, bước normalize sẽ thất bại hoặc tạo ra so sánh sai lệch (so sánh giá subcription với giá API usage là so sánh khập khiễng).

Replan:
Cần thêm 1 subtask mới: "Chuân hóa định nghĩa pricing" - quyết định 
đơn vị so sánh chung phù hợp (VD: ước tín chi phí/tháng cho use-case điển hình,
hoặc trình bày riêng 2 loại cấu trúc pricing thay vì ép về 1 con số) trước khi bước vào normalize/compare gốc.




Trong trường hợp nào Strategy B tố t hơn Strategy A?

Không phải "linh hoạt hơn nên tốt hơn" chung chung - cụ thể Strategy B đáng giá hơn khi cả 3 điều kiện sau cùng đúng:

1. Độ không chắc chắn về cấu trúc dữ liệu nguồn cao - task này đúng vì đối thủ có mô hình kinh doanh khác nhau (subsciption vs API usage), không thể biết trước cấu trúc pricing đồng nhất hay không cho đến khi thực sự research.

2. Chi phi của việc "tiếp tục sai" cao hơn chi phí replan - ở đây, viết ra 1 report so sánh hai lệnh cho stakeholder đọc có hậu quả lớn hơn nhiều so với vài lượt LLM call thêm để replan.

3. Có tín hiệu rõ ràng (observable) để phát hiện giả định sai - nếu hệ thống không có cách nào biết được assumption đã sai (VD: dữ liệu luôn trông hợp lệ dù thực chất sai), replanning cũng không cứu được - lúc đó vấn đề năm ở verification, không phải planning



1. Failure mode lớn nhất?
State Plan / Wrong Exit Condition kết hợp - Agent coi task đã đủ khi Exit Condition định nghĩa mơ hồ , dẫn tới report thiếu dữ liệu quan trọng mà không ai biết. Nguy hiểm hơn Infinite Loop vì loop ít nhất còn lộ ra ngoài (dễ phát hiện), còn Exit Condition sai thì Agent tự tin dừng và trả kết quả trông có vẻ hoàn chỉnh nhưng thực chất thiếu.

2. Làm thế nào biết hệ thống đang tốt lên?
Theo dõi đồng thời Task Success Rate tăng + Unnecessary Replanning Rate giảm + Planning Overhead ổn định hoặc giảm theo thời gian - tăng Success Rate mà Planning Overhead tăng vọt là dấu hiệu "đúng nhưng đắt", cần tối ưu Planner chứ không chỉ ăn mừng kết quả.

3. Metric nào cần theo dõi?
Ưu tiên: Task Success Rate (outcome), Unneccessary Replanning Rate (hiệu quả cơ chế), Exit Condition Precision(đúng chuẩn dừng), Plan Efficiency (chi phí).

Task Success Rate
Definition: Tỷ lệ task mà kết quả cuối cùng đúng và hoàn thành đúng ý user - bất kể có tool thừa/thiếu
hay không, miễn là kết quả cuối cùng
Formula: (số  task có final outcome đúng) / (tổng số task)

Unneccessary Replanning Rate:
Definition: Tỷ lệ replan mà plan mới không khác biệt đáng kể so với plan cũ (oscillation hoặc replan thừa)
Formula: (số lần replan mà plan mới = plan cũ) / (tổng số lần replan)

Plan Efficiency
Definition: So sánh số subtask/tool call thực tế đã dùng với số tối thiểu theo ground-truth
Formula: số subtask tối thiểu / số substask thực tế đã thực hiện

4. Có thực sự cần Agent không?
Có, nhưng chỉ ở phần lõi (research + normalize) - không cần toàn bộ pipeline

task này nên dùng Planning Agent với Replanning, vì cấu trúc dữ liệu nguồn không đồng nhất và không biết trước được (3 công ty có 3 mô hình kinh doanh, 3 cách trình bày pricing khác nhau) - đây chính xác là loại bất định mà Fixed Workflow không xử lý tốt (phải hardcode quá nhiều nhánh và vẫn sẽ gãy nếu gặp đối thủ thứ 4 có mô hình mới)

Tuy nhiên khuyến nghị không dùng replanning cho mọi phần - chỉ áp dụng ở đúng điểm cần (research + normalize, nơi dữ liệu thực sự bất định), còn phần Identify competitors và write report hoàn toàn có thể là bước cố đinh - đây là kiến trúc hybrid: Fixed Workflow ở khung ngoài, Planning Agent chỉ ở phần lõi thích ứng.

5. Có thể làm architecture đơn giản hơn không?
Có - kiên trúc lai: Fixed Window cho Identify -> research A,B,C song song -> write report, chỉ nhúng 1 Planning Agent nhỏ có replanning đúng vào bước normalize/compare - nơi thực sự cần khả năng phát hiện và sửa giả định sai.

